# Phase 3 - regenerate predictions so conformal and calibration reach the fusion ladder

**What this fixes.** `archive()` saved metrics but not predictions until part-way through
this branch, so per-split predictions exist for only `desc`, `fuse_gated`,
`fuse_gated_nograph`, `qdesc` and the Phase 0 pipeline models. Conformal prediction and
calibration both work *from* predictions, so Phase 3 currently cannot say anything about
`fuse_proposed` -- the model the whole fusion ladder is about -- or about `concat`,
`xattn`, `bilinear` and the inherited `gin_ref` reference.

Nothing about the models changes. This retrains the same five tags under the same fixed
hyper-parameters and the same splits, purely so the predictions get written this time. The
metrics it produces should reproduce the committed ones; the check at the end verifies that
rather than assuming it.

**This is safe to run while another Colab session is working.** It writes to its own Drive
folder (`mpp_phase3`), so it cannot touch the archives the end-to-end run is building, and
it needs no `--redo`.

Expected: **2-4 h on a T4** for five tags across six splits. `--resume` continues after a
timeout, so run cell 7 again as many times as it takes.

## 1. Check you actually got a GPU

In [ ]:
import torch
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print('No GPU. Runtime > Change runtime type > T4 GPU, then re-run this cell.')

## 2. Connect your Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Point at the bundle

Upload `mpp_phase3_regen.zip` (126 MB - it carries the cached ChemBERTa embeddings, which
is what makes `--seq cached` possible without running the transformer) to Drive first.

In [ ]:
BUNDLE = '/content/drive/MyDrive/mpp_phase3_regen.zip'  # edit if elsewhere
OUTDIR = '/content/drive/MyDrive/mpp_phase3'

import os
assert os.path.exists(BUNDLE), f'Not found: {BUNDLE} -- check path, re-run.'
os.makedirs(OUTDIR, exist_ok=True)
print('bundle:', round(os.path.getsize(BUNDLE)/1e6, 1), 'MB')

## 4. Unpack and install

`torchao` is uninstalled deliberately: an old build's probe *raises* instead of returning
False, which kills every run that touches `peft`.

In [ ]:
import zipfile, os

WORK = '/content/mpp'
os.makedirs(WORK, exist_ok=True)
with zipfile.ZipFile(BUNDLE) as z:
    z.extractall(WORK)
os.chdir(WORK)

!pip -q install peft accelerate torch_geometric
!pip -q uninstall -y torchao

import torch_geometric
print('torch_geometric', torch_geometric.__version__)

## 5. Keep results on Drive

`results/runs` becomes a symlink into Drive, so a disconnect loses nothing and `--resume`
can see what already finished. Note this is a **different folder** from the end-to-end
run's, so the two cannot collide.

In [ ]:
import os

os.makedirs(f'{OUTDIR}/runs', exist_ok=True)
os.makedirs('results', exist_ok=True)
if not os.path.islink('results/runs'):
    if os.path.exists('results/runs'):
        import shutil; shutil.rmtree('results/runs')
    os.symlink(f'{OUTDIR}/runs', 'results/runs')
print('results/runs ->', os.path.realpath('results/runs'))

## 6. Smoke test - do not skip this

One epoch on the smallest dataset. If the cached-embedding path or the graph encoder is
broken on this runtime, it fails here in a minute rather than three hours in. The smoke
outputs are deleted immediately so they cannot be mistaken for results.

In [ ]:
!python -m src.data.materialize --variant deepchem --artifacts chemberta ecfp graphs desc
!python -m src.train.train_fusion --mode proposed --tag _smoke --seq cached --datasets freesolv --epochs 1 --patience 1 --device cuda
!rm -f results/metrics/*_smoke*.csv results/preds/*_smoke*.npy
!rm -f models/*_smoke*.pt
print('smoke test done and cleaned up')

## 7. Train

`--seq cached` is what makes this the *frozen* setting, matching the committed ladder --
not the end-to-end one. `--resume` skips whatever already finished, so re-running this cell
after a timeout continues rather than restarting.

`fuse_gated` is deliberately absent: it already has archived predictions, and re-running it
would only risk what is already good.

In [ ]:
!python -m scripts.run_view_multiseed     --tags fuse_concat fuse_xattn fuse_bilinear fuse_proposed gin_ref     --variants deepchem seed0 seed1 seed2 seed3 seed4     --artifacts chemberta ecfp graphs desc     --seq cached --device cuda --resume --restore none

## 8. Check what finished

In [ ]:
import glob
tags = ['fuse_concat','fuse_xattn','fuse_bilinear','fuse_proposed','gin_ref']
allok = True
for v in ['deepchem','seed0','seed1','seed2','seed3','seed4']:
    m = [len(glob.glob(f'{OUTDIR}/runs/{v}/metrics/*_{t}_*.csv')) for t in tags]
    p = [len(glob.glob(f'{OUTDIR}/runs/{v}/preds/*_{t}_*.npy')) for t in tags]
    done = all(c == 16 for c in m) and all(c == 16 for c in p)
    allok &= done
    print(v, 'metrics', dict(zip(tags, m)), 'preds', dict(zip(tags, p)),
          'ok' if done else 'INCOMPLETE - re-run cell 7')
print()
print('ALL DONE - run cell 9' if allok else 'Not finished yet. Re-run cell 7.')

## 9. Bring the results home

Then, locally and **with `--dry-run` first**:

```
python -m scripts.merge_colab_results ~/Downloads/phase3_results.zip --dry-run
```

Unzipping by hand nests `results/{metrics,preds,figs,logs}` inside `results/runs/`, which
reads as a mass deletion. The merge script exists to prevent exactly that.

In [ ]:
import shutil, os
out = shutil.make_archive('/content/phase3_results', 'zip', f'{OUTDIR}/runs')
print('wrote', out, round(os.path.getsize(out)/1e6, 2), 'MB')
from google.colab import files
files.download(out)